# PIPE 1 — Train & Package SciBERT-SolarPhysics-Search (DAPT + Contrastive FT) + Index Build

End-to-end pipeline: dataset preparation → domain-adaptive pretraining (MLM) → supervised contrastive fine-tuning → embedding export → FAISS index build → (optional) Hugging Face upload.

## Run order
- 0) Setup & configuration
- 1) Load corpus export (title/abstract/keywords) and build training datasets
- 2) Domain-Adaptive Pretraining (MLM / DAPT)
- 3) Supervised Contrastive Fine-Tuning (optional)
- 4) Export embeddings + build FAISS index
- 5) Quick retrieval sanity checks
- 6) Optional: push model artifacts to Hugging Face

## Notes
- This release contains **no embedded secrets** and **no stored outputs**.
- Configure paths and keys via environment variables or `.env`.


## Inputs expected
- `data/corpus.parquet` (or set `CORPUS_PATH`) with columns: `doc_id`, `title`, `abstract` (optional: `keywords`).
- Optional supervised pairs file `data/pairs.jsonl` (or `PAIRS_PATH`) for contrastive FT.

## Outputs
- `artifacts/pipe1/model/` (trained encoder)
- `artifacts/pipe1/embeddings/embeddings.npy`
- `artifacts/retriever/faiss.index` + `artifacts/retriever/docs.parquet`


In [ ]:
# ============================================================
# 0) Setup & configuration
# ============================================================
import os
from pathlib import Path
import random
import numpy as np

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
ART_PIPE1 = ROOT / "artifacts" / "pipe1"
ART_MODEL = ART_PIPE1 / "model"
ART_EMB   = ART_PIPE1 / "embeddings"
ART_RET   = ROOT / "artifacts" / "retriever"

for p in [DATA_DIR, ART_PIPE1, ART_MODEL, ART_EMB, ART_RET]:
    p.mkdir(parents=True, exist_ok=True)

CORPUS_PATH = Path(os.getenv("CORPUS_PATH", str(DATA_DIR / "corpus.parquet")))
PAIRS_PATH  = Path(os.getenv("PAIRS_PATH",  str(DATA_DIR / "pairs.jsonl")))  # optional

print("ROOT:", ROOT)
print("CORPUS_PATH:", CORPUS_PATH)
print("PAIRS_PATH:", PAIRS_PATH)

In [ ]:
# ============================================================
# 1) Load corpus and build training datasets
# ============================================================
import pandas as pd

assert CORPUS_PATH.exists(), (
    f"Corpus file not found at {CORPUS_PATH}. "
    "Place it under data/ or set CORPUS_PATH."
)

df = pd.read_parquet(CORPUS_PATH) if CORPUS_PATH.suffix.lower()==".parquet" else pd.read_csv(CORPUS_PATH)
required = {"doc_id","title","abstract"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns in corpus: {missing}"

if "keywords" in df.columns:
    df["text"] = df["title"].fillna("") + "\n\n" + df["abstract"].fillna("") + "\n\n" + df["keywords"].fillna("")
else:
    df["text"] = df["title"].fillna("") + "\n\n" + df["abstract"].fillna("")

texts_for_dapt = df["text"].astype(str).tolist()
print("Docs:", len(df), "| DAPT texts:", len(texts_for_dapt))

pairs = None
if PAIRS_PATH.exists():
    import json
    pairs = []
    with open(PAIRS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            obj=json.loads(line)
            if "anchor" in obj and "positive" in obj:
                pairs.append(obj)
    print("Loaded supervised pairs:", len(pairs))
else:
    print("PAIRS_PATH not found — contrastive FT will be skipped unless you provide pairs.jsonl.")

In [ ]:
# ============================================================
# 2) Domain-Adaptive Pretraining (DAPT) — MLM (Hugging Face Trainer)
# ============================================================
import torch
from transformers import (AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling,
                          Trainer, TrainingArguments)
from datasets import Dataset

BASE_MODEL = os.getenv("BASE_MODEL", "allenai/scibert_scivocab_uncased")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)

max_length = int(os.getenv("MAX_LENGTH_MLM", "256"))
train_frac = float(os.getenv("DAPT_TRAIN_FRAC", "0.98"))

ds = Dataset.from_dict({"text": texts_for_dapt})
ds = ds.train_test_split(test_size=(1-train_frac), seed=SEED)

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)

ds_tok = ds.map(tok_fn, batched=True, remove_columns=["text"])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

dapt_out = ART_MODEL / "dapt_scibert_solarphysics"
dapt_out.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(dapt_out),
    overwrite_output_dir=True,
    num_train_epochs=float(os.getenv("DAPT_EPOCHS", "5")),
    per_device_train_batch_size=int(os.getenv("DAPT_BATCH", "32")),
    per_device_eval_batch_size=int(os.getenv("DAPT_BATCH", "32")),
    learning_rate=float(os.getenv("DAPT_LR", "2e-5")),
    evaluation_strategy="steps",
    eval_steps=int(os.getenv("DAPT_EVAL_STEPS", "500")),
    save_steps=int(os.getenv("DAPT_SAVE_STEPS", "500")),
    save_total_limit=2,
    logging_steps=int(os.getenv("DAPT_LOG_STEPS", "100")),
    seed=SEED,
    report_to=[],
)

trainer = Trainer(
    model=mlm_model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["test"],
    data_collator=collator,
)

# IMPORTANT: Training can take hours depending on your corpus and hardware.
# Run locally/on a GPU instance when you are ready:
# trainer.train()

print("DAPT prepared. Uncomment `trainer.train()` to start training.")
print("DAPT artifacts dir:", dapt_out)

In [ ]:
# ============================================================
# 3) Supervised Contrastive Fine-Tuning (optional)
# ============================================================
import torch
from torch import nn
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

ENCODER_PATH = os.getenv("ENCODER_PATH", str(dapt_out))
try:
    encoder = AutoModel.from_pretrained(ENCODER_PATH)
    print("Loaded encoder from:", ENCODER_PATH)
except Exception as e:
    encoder = AutoModel.from_pretrained(BASE_MODEL)
    print("Falling back to base encoder:", BASE_MODEL, "| reason:", str(e)[:200])

encoder.to(device)
encoder.train()

def mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.05):
        super().__init__()
        self.tau = temperature

    def forward(self, z_a, z_p):
        z_a = nn.functional.normalize(z_a, p=2, dim=1)
        z_p = nn.functional.normalize(z_p, p=2, dim=1)
        logits = (z_a @ z_p.T) / self.tau
        labels = torch.arange(z_a.size(0), device=z_a.device)
        return nn.CrossEntropyLoss()(logits, labels)

if not pairs:
    print("No supervised pairs loaded — skipping contrastive fine-tuning.")
else:
    lr = float(os.getenv("CONTRASTIVE_LR", "2e-5"))
    epochs = int(os.getenv("CONTRASTIVE_EPOCHS", "10"))
    batch = int(os.getenv("CONTRASTIVE_BATCH", "32"))
    max_len = int(os.getenv("MAX_LENGTH_CONTRAST", "256"))
    loss_fn = SupConLoss(temperature=float(os.getenv("TAU", "0.05"))).to(device)
    opt = torch.optim.AdamW(encoder.parameters(), lr=lr)

    def embed_text(batch_texts):
        tok = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
        out = encoder(**tok)
        emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
        return emb

    from math import ceil
    steps = ceil(len(pairs)/batch)

    for ep in range(1, epochs+1):
        random.shuffle(pairs)
        total = 0.0
        for s in range(0, len(pairs), batch):
            chunk = pairs[s:s+batch]
            anchors = [c["anchor"] for c in chunk]
            positives = [c["positive"] for c in chunk]
            z_a = embed_text(anchors)
            z_p = embed_text(positives)

            loss = loss_fn(z_a, z_p)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss.detach().cpu())
        print(f"Epoch {ep:02d}/{epochs} | loss={total/steps:.4f}")

    final_out = ART_MODEL / "scibert_solarphysics_search"
    final_out.mkdir(parents=True, exist_ok=True)
    encoder.save_pretrained(final_out)
    tokenizer.save_pretrained(final_out)
    print("Saved contrastive-finetuned encoder to:", final_out)

In [ ]:
# ============================================================
# 4) Export embeddings + build FAISS index
# ============================================================
import numpy as np
import faiss
import torch

FINAL_ENCODER_DIR = ART_MODEL / "scibert_solarphysics_search"
enc = encoder
enc.eval()

texts = df["text"].astype(str).tolist()
BATCH = int(os.getenv("EMB_BATCH", "16"))
MAX_LEN = int(os.getenv("MAX_LENGTH_EMB", "256"))

embs = []
with torch.no_grad():
    for i in range(0, len(texts), BATCH):
        bt = texts[i:i+BATCH]
        tok = tokenizer(bt, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
        out = enc(**tok)
        pooled = mean_pool(out.last_hidden_state, tok["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        embs.append(pooled.cpu().numpy())

embs = np.vstack(embs).astype("float32")
print("Embeddings:", embs.shape)

emb_path = ART_EMB / "embeddings.npy"
np.save(emb_path, embs)
print("Saved:", emb_path)

docs_path = ART_RET / "docs.parquet"
df[["doc_id","title","abstract"]].to_parquet(docs_path, index=False)
print("Saved:", docs_path)

dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embs)

faiss_path = ART_RET / "faiss.index"
faiss.write_index(index, str(faiss_path))
print("Saved:", faiss_path)

In [ ]:
# ============================================================
# 5) Quick retrieval sanity check
# ============================================================
import pandas as pd
import faiss
import torch

index = faiss.read_index(str(ART_RET / "faiss.index"))
docs  = pd.read_parquet(ART_RET / "docs.parquet")

query = os.getenv("QUERY", "physics-informed machine learning for solar flare forecasting")

enc.eval()
with torch.no_grad():
    tok = tokenizer([query], padding=True, truncation=True, max_length=256, return_tensors="pt").to(device)
    out = enc(**tok)
    q_emb = mean_pool(out.last_hidden_state, tok["attention_mask"])
    q_emb = torch.nn.functional.normalize(q_emb, p=2, dim=1).cpu().numpy().astype("float32")

D, I = index.search(q_emb, 5)
print("Query:", query)
for r, (idx, score) in enumerate(zip(I[0], D[0]), start=1):
    row = docs.iloc[int(idx)]
    print(f"{r:02d} score={float(score):.4f} | {row['title']}")

## 6) Optional: Hugging Face upload
This repository does **not** store credentials. If you want to publish artifacts:
1) `huggingface-cli login`
2) Push `artifacts/pipe1/model/scibert_solarphysics_search/` using `huggingface_hub`.
